# Functional evaluation of LLM outputs

We will use several strategies to evaluate if the LLM produces a 'correct' WDL workflow, not merely one that will run. We want to know if the WDL will perform the tasks the user has requested.

We will employ reference-based evaluation using pre-built WILDS WDL pipelines as our ground truth. We will have five prompt-ground truth pairs, using these pipelines:
- bwa-gatk (basic)
- sra-star (basic)
- sra-salmon (basic)
- star-deseq2 (intermediate)
- saturation (intermediate)

Other thoughts:
- Don't have the model output meta/parameter meta sections because that makes eval harder and that stuff isn't functional anyway

Separately, we should confirm that WDL tasks being retrieved from our RAG database will indeed meet the users request. We expect that they always will because we will use keyword filtering based on user input terms but we should still confirm this (and confirm we can choose between equivalent tasks).

## Imports

In [1]:
from rapidfuzz.distance import Levenshtein

## Lexical (text) similarity

Use the simple Levenshtein distance because code-specific textual similarity metrics rely on proper parsing of WDL and there isn't a lot of tooling for WDL syntax yet.

In [ ]:
def lexical_similarity(ref, to_eval):
    score = 1 - Levenshtein.normalized_distance(ref, to_eval)
    return score

0.8918918918918919

Let's say it has to have a score of >= 0.85 to pass.

In [3]:
pass_score_lexical = 0.85

truth="The Eiffel Tower is located in Paris."
generated="The Eiffel Tower is located in India."

lexical_score = lexical_similarity(ref=truth, to_eval=generated)
lexical_score >= pass_score_lexical

True

## Semantic (meaning) similarity

Measure cosine similarity of codeBERT embeddings between ground truth and generated WDL. While codeBERT was not trained on WDL (or Rust, or many other langauges for that matter), it should be able to 'understand' the syntax enough to work.

In [4]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import numpy as np

embed_model = HuggingFaceEmbedding(model_name="microsoft/codebert-base")

def semantic_similarity(ref, to_eval):
    # Get embeddings of the text we're comparing
    emb_ref = np.array(embed_model.get_text_embedding(ref))
    emb_eval = np.array(embed_model.get_text_embedding(to_eval))
    # Calculate cosine similarity with numpy
    product_of_lengths = (np.linalg.norm(emb_ref) * np.linalg.norm(emb_eval))
    dot_product = np.dot(emb_ref, emb_eval)
    score = float(dot_product / product_of_lengths)

    return score

The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 60212.56it/s]


Let's say it has to have a score of >= 0.90 to pass

In [5]:
pass_score_semantic = 0.9

truth="The Eiffel Tower is located in Paris."
generated="The Eiffel Tower is located in India."

semantic_score = semantic_similarity(ref=truth, to_eval=generated)
semantic_score >= pass_score_semantic

True

## Rule-based metric

Ensure that all retrieved WDL tasks and only those retrieved WDL tasks are used in the final output.

Note: This assuemes that the retrieval was done correctly (that needs to be evaluated separately)

In [ ]:
# TODO